# Experimenting with Data

Taking the data and reading it 

In [25]:
import zipfile
with zipfile.ZipFile("default+of+credit+card+clients.zip") as z:
    z.printdir()          # lists the files [40]

File Name                                             Modified             Size
default of credit card clients.xls             2023-05-22 15:22:14      5539328


In [26]:

import pandas as pd
with zipfile.ZipFile("default+of+credit+card+clients.zip") as z:
    with z.open("default of credit card clients.xls") as f:
        df = pd.read_excel(f, header=1)   # row 0 is a junk label row

In [27]:
df = df.rename(columns={"default payment next month": "default"})
df = df.drop(columns="ID")
print(df.shape, df.dtypes, df.isna().sum())
df.to_csv("credit_default.csv", index=False)   # faster to reload later

(30000, 24) LIMIT_BAL    int64
SEX          int64
EDUCATION    int64
MARRIAGE     int64
AGE          int64
PAY_0        int64
PAY_2        int64
PAY_3        int64
PAY_4        int64
PAY_5        int64
PAY_6        int64
BILL_AMT1    int64
BILL_AMT2    int64
BILL_AMT3    int64
BILL_AMT4    int64
BILL_AMT5    int64
BILL_AMT6    int64
PAY_AMT1     int64
PAY_AMT2     int64
PAY_AMT3     int64
PAY_AMT4     int64
PAY_AMT5     int64
PAY_AMT6     int64
default      int64
dtype: object LIMIT_BAL    0
SEX          0
EDUCATION    0
MARRIAGE     0
AGE          0
PAY_0        0
PAY_2        0
PAY_3        0
PAY_4        0
PAY_5        0
PAY_6        0
BILL_AMT1    0
BILL_AMT2    0
BILL_AMT3    0
BILL_AMT4    0
BILL_AMT5    0
BILL_AMT6    0
PAY_AMT1     0
PAY_AMT2     0
PAY_AMT3     0
PAY_AMT4     0
PAY_AMT5     0
PAY_AMT6     0
default      0
dtype: int64


# Markov Chain Prediction Model
the file contains data labled PAY_0 through PAY_6 which gives information on payments from previous months. It is most beneficial to be able to predict defaulting on payments earlier on rather than later on.

In [28]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix

data = pd.read_csv("credit_default.csv")

# Chronological order: oldest payment status to most recent.
pay_cols = ["PAY_6", "PAY_5", "PAY_4", "PAY_3", "PAY_2", "PAY_0"]
states = ["current", "1_month_late", "2_months_late", "3plus_months_late"]

def to_state(status):
    if status <= 0:
        return "current"
    if status == 1:
        return "1_month_late"
    if status == 2:
        return "2_months_late"
    return "3plus_months_late"

payment_states = data[pay_cols].apply(lambda col: col.map(to_state))
y = data["default"].astype(int)

train_idx, test_idx = train_test_split(
    np.arange(len(data)), test_size=0.25, random_state=42, stratify=y
)

# Estimate the transition matrix using training clients only.
transition_counts = pd.DataFrame(1.0, index=states, columns=states)  # Laplace smoothing
for idx in train_idx:
    sequence = payment_states.iloc[idx].tolist()
    for source, destination in zip(sequence[:-1], sequence[1:]):
        transition_counts.loc[source, destination] += 1

transition_matrix = transition_counts.div(transition_counts.sum(axis=1), axis=0)

# Smoothed default rate for each observed payment state, from training data.
train_states = payment_states.loc[train_idx, "PAY_0"]
global_default_rate = y.iloc[train_idx].mean()
state_default_rate = {}

for state in states:
    mask = train_states.eq(state)
    # 20 prior observations shrink estimates for rare states toward the overall rate.
    state_default_rate[state] = (
        y.iloc[train_idx][mask].sum() + 20 * global_default_rate
    ) / (mask.sum() + 20)

# Project one Markov transition from each client's latest payment state.
test_current_states = payment_states.loc[test_idx, "PAY_0"]
risk_by_state = pd.Series(state_default_rate)
predicted_default_probability = (
    transition_matrix.loc[test_current_states].to_numpy() @ risk_by_state[states].to_numpy()
)

y_test = y.iloc[test_idx].to_numpy()
print("Transition matrix:")
display(transition_matrix)
print(f"ROC AUC: {roc_auc_score(y_test, predicted_default_probability):.3f}")
print(f"Accuracy (0.5 threshold): {accuracy_score(y_test, predicted_default_probability >= 0.5):.3f}")
print("Confusion matrix:")
print(confusion_matrix(y_test, predicted_default_probability >= 0.5))

Transition matrix:


,current,1_month_late,2_months_late,3plus_months_late
current,0.938298,0.014444,0.047248,0.000010
1_month_late,0.029412,0.911765,0.029412,0.029412
2_months_late,0.255654,0.102556,0.580060,0.061729
3plus_months_late,0.112163,0.084487,0.286963,0.516387


ROC AUC: 0.695
Accuracy (0.5 threshold): 0.820
Confusion matrix:
[[5597  244]
 [1107  552]]


For Current and 1 month late the probaility of staying the same is very high (over 90% for each) 2 and over three months late on the other hand result in more varying changes. 3 

In [29]:
#only using data from the oldest 3 months
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix

data = pd.read_csv("credit_default.csv")
states = ["current", "1_month_late", "2_months_late", "3plus_months_late"]

def to_state(s):
    return states[0] if s <= 0 else states[1] if s == 1 else states[2] if s == 2 else states[3]

y = data["default"].astype(int)
train_idx, test_idx = train_test_split(
    np.arange(len(data)), test_size=0.25, random_state=42, stratify=y
)

def run_markov(pay_cols, prior_n=20):
    ps = data[pay_cols].apply(lambda c: c.map(to_state))
    latest = pay_cols[-1]

    train = ps.iloc[train_idx]
    counts = pd.DataFrame(1.0, index=states, columns=states)      # Laplace smoothing
    for a, b in zip(pay_cols[:-1], pay_cols[1:]):
        counts += pd.crosstab(train[a], train[b]).reindex(index=states, columns=states, fill_value=0)
    P = counts.div(counts.sum(axis=1), axis=0)

    y_tr = y.iloc[train_idx]
    g = y_tr.mean()
    risk = pd.Series({
        s: (y_tr[train[latest].eq(s)].sum() + prior_n * g) / (train[latest].eq(s).sum() + prior_n)
        for s in states
    })

    test_states = ps.iloc[test_idx][latest]
    prob = P.loc[test_states].to_numpy() @ risk[states].to_numpy()
    y_test = y.iloc[test_idx].to_numpy()

    thr = g                                   # base-rate threshold
    return {
        "columns": pay_cols,
        "auc": roc_auc_score(y_test, prob),
        "acc@0.5": accuracy_score(y_test, prob >= 0.5),
        f"acc@{thr:.2f}": accuracy_score(y_test, prob >= thr),
        "cm@base_rate": confusion_matrix(y_test, prob >= thr),
        "P": P,
    }

all_months = ["PAY_6", "PAY_5", "PAY_4", "PAY_3", "PAY_2", "PAY_0"]
oldest3    = ["PAY_6", "PAY_5", "PAY_4"]

res_all = run_markov(all_months)
res_old = run_markov(oldest3)

summary = pd.DataFrame({
    "all 6 months":    {k: v for k, v in res_all.items() if k in ("auc", "acc@0.5")},
    "oldest 3 months": {k: v for k, v in res_old.items() if k in ("auc", "acc@0.5")},
})
display(summary)
display(res_old["P"]); display(res_old["cm@base_rate"])

,all 6 months,oldest 3 months
auc,0.695464,0.606198
acc@0.5,0.819867,0.784000


,current,1_month_late,2_months_late,3plus_months_late
current,0.960521,0.000049,0.039405,0.000025
1_month_late,0.250000,0.250000,0.250000,0.250000
2_months_late,0.300518,0.000493,0.648409,0.050580
3plus_months_late,0.116183,0.002075,0.271784,0.609959


array([[5436,  405],
       [1194,  465]])

In [30]:
#two step Markov model using the oldest 3 months of data and all 6
def run_markov2(pay_cols, alpha=1.0, prior_n=20):
    ps = data[pay_cols].apply(lambda c: c.map(to_state))
    train, test = ps.iloc[train_idx], ps.iloc[test_idx]
    prev, last = pay_cols[-2], pay_cols[-1]

    pairs = pd.MultiIndex.from_product([states, states])
    counts = pd.DataFrame(alpha, index=pairs, columns=states)
    for c0, c1, c2 in zip(pay_cols[:-2], pay_cols[1:-1], pay_cols[2:]):
        ct = pd.crosstab([train[c0], train[c1]], train[c2])
        counts += ct.reindex(index=pairs, columns=states, fill_value=0)
    P2 = counts.div(counts.sum(axis=1), axis=0)

    y_tr = y.iloc[train_idx]
    g = y_tr.mean()
    grp = y_tr.groupby([train[prev], train[last]]).agg(["sum", "count"]).reindex(pairs, fill_value=0)
    risk2 = (grp["sum"] + prior_n * g) / (grp["count"] + prior_n)

    R = np.array([[risk2.loc[(b, k)] for k in states] for (a, b) in pairs])
    pair_prob = pd.Series((P2.to_numpy() * R).sum(axis=1), index=pairs)

    te_pairs = pd.MultiIndex.from_arrays([test[prev], test[last]])
    prob = pair_prob.reindex(te_pairs).to_numpy()
    y_test = y.iloc[test_idx].to_numpy()

    thr = g
    return {"auc": roc_auc_score(y_test, prob),
            "acc@0.5": accuracy_score(y_test, prob >= 0.5),
            "recall@base": recall_score(y_test, prob >= thr),
            "cm@base_rate": confusion_matrix(y_test, prob >= thr),
            "P2": P2, "prob": prob, "y_test": y_test}

from sklearn.metrics import recall_score
res2_all = run_markov2(["PAY_6","PAY_5","PAY_4","PAY_3","PAY_2","PAY_0"])
res2_old = run_markov2(["PAY_6","PAY_5","PAY_4"])
print(res2_all["auc"], res2_old["auc"])

0.6911376306355923 0.6008399294174879


In [31]:
#adding a 5-state model to see if it improves performance
from sklearn.metrics import recall_score

states5 = ["current", "1_late", "2_late", "3_late", "4plus_late"]

def to_state5(s):
    return states5[0] if s <= 0 else states5[min(int(s), 4)]

def run_markov_n(pay_cols, states, to_state_fn, prior_n=20):
    ps = data[pay_cols].apply(lambda c: c.map(to_state_fn))
    train = ps.iloc[train_idx]
    latest = pay_cols[-1]

    counts = pd.DataFrame(1.0, index=states, columns=states)
    for a, b in zip(pay_cols[:-1], pay_cols[1:]):
        counts += pd.crosstab(train[a], train[b]).reindex(index=states, columns=states, fill_value=0)
    P = counts.div(counts.sum(axis=1), axis=0)

    y_tr = y.iloc[train_idx]
    g = y_tr.mean()
    risk = pd.Series({
        s: (y_tr[train[latest].eq(s)].sum() + prior_n * g) / (train[latest].eq(s).sum() + prior_n)
        for s in states
    })

    test_states = ps.iloc[test_idx][latest]
    prob = P.loc[test_states].to_numpy() @ risk[states].to_numpy()
    y_test = y.iloc[test_idx].to_numpy()
    return {"auc": roc_auc_score(y_test, prob),
            "acc@0.5": accuracy_score(y_test, prob >= 0.5),
            "recall@base": recall_score(y_test, prob >= g),
            "cm@base_rate": confusion_matrix(y_test, prob >= g),
            "P": P, "risk": risk, "prob": prob, "y_test": y_test}

all_months = ["PAY_6","PAY_5","PAY_4","PAY_3","PAY_2","PAY_0"]
res5 = run_markov_n(all_months, states5, to_state5)
print(res5["auc"], res5["recall@base"])
display(res5["P"]); display(res5["risk"])

0.6954623006972288 0.5081374321880651


,current,1_late,2_late,3_late,4plus_late
current,0.938288,0.014444,0.047247,0.000010,0.000010
1_late,0.028571,0.885714,0.028571,0.028571,0.028571
2_late,0.255634,0.102547,0.580013,0.061724,0.000082
3_late,0.161871,0.095923,0.335731,0.155875,0.250600
4plus_late,0.036697,0.067890,0.211009,0.045872,0.638532


current       0.137755
1_late        0.342714
2_late        0.688725
3_late        0.723049
4plus_late    0.532430
dtype: float64